# 01 - Sequential transfer learning with YOLOv5x

This notebook implements the finalized pipeline:

```text
COCO-pretrained YOLOv5x
-> HRPlanesv2 fine-tuning
-> HRPlanesv2 validation selects Stage-1 best.pt
-> HRPlanesv2 held-out test
-> CORS-ADD-HBB fine-tuning from Stage-1 best.pt
-> internal validation selects final best.pt
-> official val2017, reserved as held-out final test
```

Run notebook 00 first in the same Colab runtime. Formal training cells are guarded by explicit Boolean switches so that opening or running setup cells cannot accidentally start a long GPU job.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path('/content/drive/MyDrive/RS_PRO')
WORK_ROOT = Path('/content/rs_pro_work')
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
RUNS_ROOT = ARTIFACT_ROOT / 'runs'
EVALUATION_ROOT = ARTIFACT_ROOT / 'evaluation'

YOLOV5_ROOT = Path('/content/yolov5')
YOLOV5_REF = 'v7.0'

HR_YAML = WORK_ROOT / 'configs' / 'hrplanesv2.yaml'
CORS_YAML = WORK_ROOT / 'configs' / 'cors_add_hbb.yaml'

IMAGE_SIZE = 640
BATCH_SIZE = 4
EPOCHS_STAGE1 = 100
EPOCHS_STAGE2 = 100
PATIENCE = 20
WORKERS = 2
RANDOM_SEED = 42

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)

for required in [PROJECT_ROOT, HR_YAML, CORS_YAML]:
    if not required.exists():
        raise FileNotFoundError(
            f'Missing {required}. Run notebook 00 in this Colab runtime first.'
        )

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Python:', sys.version)


## Install the official YOLOv5 implementation


In [ ]:
if not YOLOV5_ROOT.exists():
    subprocess.run(
        [
            'git', 'clone', '--depth', '1', '--branch', YOLOV5_REF,
            'https://github.com/ultralytics/yolov5.git',
            str(YOLOV5_ROOT),
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(YOLOV5_ROOT / 'requirements.txt')],
    check=True,
)

commit = subprocess.run(
    ['git', '-C', str(YOLOV5_ROOT), 'rev-parse', 'HEAD'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

environment = {
    'yolov5_ref': YOLOV5_REF,
    'yolov5_commit': commit,
    'python': sys.version,
    'pytorch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'random_seed': RANDOM_SEED,
}

(ARTIFACT_ROOT / 'environment.json').write_text(
    json.dumps(environment, indent=2), encoding='utf-8'
)
print(json.dumps(environment, indent=2))


## Stage 1 - Fine-tune COCO-pretrained YOLOv5x on HRPlanesv2

`train.py` evaluates the HRPlanesv2 validation split after every epoch and writes `best.pt` based on validation fitness. The test split is not used by this command.


In [ ]:
RUN_STAGE1_TRAINING = False  # Change to True when ready to start formal training.

stage1_name = 'stage1_hrplanesv2_yolov5x'
stage1_dir = RUNS_ROOT / stage1_name

stage1_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'train.py'),
    '--weights', 'yolov5x.pt',
    '--data', str(HR_YAML),
    '--img', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS_STAGE1),
    '--patience', str(PATIENCE),
    '--workers', str(WORKERS),
    '--seed', str(RANDOM_SEED),
    '--save-period', '10',
    '--project', str(RUNS_ROOT),
    '--name', stage1_name,
    '--exist-ok',
]

print(' '.join(stage1_command))
if RUN_STAGE1_TRAINING:
    subprocess.run(stage1_command, cwd=YOLOV5_ROOT, check=True)
else:
    print('Stage 1 was not started. Set RUN_STAGE1_TRAINING = True when ready.')


## Inspect Stage-1 training history


In [ ]:
stage1_results = stage1_dir / 'results.csv'
if stage1_results.exists():
    stage1_history = pd.read_csv(stage1_results)
    display(stage1_history.tail())
else:
    print('No Stage-1 results yet:', stage1_results)


## Evaluate the selected Stage-1 model on the HRPlanesv2 test split

Run this only after Stage 1 has completed. The resulting metrics describe the HRPlanesv2 stage and must not be used to select the checkpoint.


In [ ]:
RUN_HR_TEST = False  # Change to True only after Stage 1 is complete.

HR_BEST = stage1_dir / 'weights' / 'best.pt'

hr_test_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'val.py'),
    '--weights', str(HR_BEST),
    '--data', str(HR_YAML),
    '--task', 'test',
    '--img', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--workers', str(WORKERS),
    '--save-txt',
    '--save-conf',
    '--save-json',
    '--project', str(EVALUATION_ROOT),
    '--name', 'hrplanesv2_stage1_test',
    '--exist-ok',
]

print(' '.join(hr_test_command))
if RUN_HR_TEST:
    if not HR_BEST.exists():
        raise FileNotFoundError(f'Stage-1 best.pt not found: {HR_BEST}')
    subprocess.run(hr_test_command, cwd=YOLOV5_ROOT, check=True)
else:
    print('HRPlanesv2 test was not started.')


## Stage 2 - Fine-tune the Stage-1 model on CORS-ADD-HBB

This stage must initialize from the HRPlanesv2 `best.pt`, not from the original COCO weight. The internal CORS validation split selects the final checkpoint. The official `val2017` images remain untouched during training.


In [ ]:
RUN_STAGE2_TRAINING = False  # Change to True after Stage 1 is complete.

stage2_name = 'stage2_cors_add_hbb_yolov5x'
stage2_dir = RUNS_ROOT / stage2_name

stage2_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'train.py'),
    '--weights', str(HR_BEST),
    '--data', str(CORS_YAML),
    '--img', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS_STAGE2),
    '--patience', str(PATIENCE),
    '--workers', str(WORKERS),
    '--seed', str(RANDOM_SEED),
    '--save-period', '10',
    '--project', str(RUNS_ROOT),
    '--name', stage2_name,
    '--exist-ok',
]

print(' '.join(stage2_command))
if RUN_STAGE2_TRAINING:
    if not HR_BEST.exists():
        raise FileNotFoundError(f'Stage-1 best.pt not found: {HR_BEST}')
    subprocess.run(stage2_command, cwd=YOLOV5_ROOT, check=True)
else:
    print('Stage 2 was not started. Set RUN_STAGE2_TRAINING = True when ready.')


## Inspect Stage-2 training history


In [ ]:
stage2_results = stage2_dir / 'results.csv'
if stage2_results.exists():
    stage2_history = pd.read_csv(stage2_results)
    display(stage2_history.tail())
else:
    print('No Stage-2 results yet:', stage2_results)


## Final held-out CORS-ADD-HBB test

The CORS YAML maps the original `val2017` release folder to `test`. Run this cell once after the final Stage-2 checkpoint has been selected. Do not use these results to change training settings or choose a different checkpoint.


In [ ]:
RUN_FINAL_CORS_TEST = False  # Change to True only after Stage 2 is complete.

CORS_BEST = stage2_dir / 'weights' / 'best.pt'

cors_test_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'val.py'),
    '--weights', str(CORS_BEST),
    '--data', str(CORS_YAML),
    '--task', 'test',
    '--img', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--workers', str(WORKERS),
    '--save-txt',
    '--save-conf',
    '--save-json',
    '--project', str(EVALUATION_ROOT),
    '--name', 'cors_add_hbb_final_test',
    '--exist-ok',
]

print(' '.join(cors_test_command))
if RUN_FINAL_CORS_TEST:
    if not CORS_BEST.exists():
        raise FileNotFoundError(f'Final best.pt not found: {CORS_BEST}')
    subprocess.run(cors_test_command, cwd=YOLOV5_ROOT, check=True)
else:
    print('Final CORS-ADD-HBB test was not started.')


## Resume interrupted training


In [ ]:
# Example only. Set the path to the relevant last.pt and change RUN_RESUME to True.
RUN_RESUME = False
RESUME_CHECKPOINT = stage2_dir / 'weights' / 'last.pt'

resume_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'train.py'),
    '--resume', str(RESUME_CHECKPOINT),
]

print(' '.join(resume_command))
if RUN_RESUME:
    if not RESUME_CHECKPOINT.exists():
        raise FileNotFoundError(RESUME_CHECKPOINT)
    subprocess.run(resume_command, cwd=YOLOV5_ROOT, check=True)


## Outputs

Important outputs remain in Google Drive:

- `RS_PRO/artifacts/runs/stage1_hrplanesv2_yolov5x/weights/best.pt`
- `RS_PRO/artifacts/runs/stage2_cors_add_hbb_yolov5x/weights/best.pt`
- `RS_PRO/artifacts/evaluation/hrplanesv2_stage1_test/`
- `RS_PRO/artifacts/evaluation/cors_add_hbb_final_test/`

Continue with notebook 02 after the final Stage-2 model is available.
